### Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

### Read Bronze data

In [0]:
df = spark.table("workspace.bronze.crm_cust_info")
display(df)

## Silver Transformation

### Trimming

In [0]:
df.count()

### Normalization

In [0]:
df = (
    df
    .withColumn(
        "cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status")) == "S", "Single")
         .when(F.upper(F.col("cst_marital_status")) == "M", "Married")
         .otherwise("n/a")
    )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr")) == "F", "Female")
         .when(F.upper(F.col("cst_gndr")) == "M", "Male")
         .otherwise("n/a")
    )
)

In [0]:
df.select("cst_gndr").distinct().show()

### Remove Records with Missing Customer ID

In [0]:
df = df.filter(col("cst_id").isNotNull())
df.count()


### Rename Columns

In [0]:
rename_cols = {
    "cst_id":"customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}

for old_name,new_name in rename_cols.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
df.show()

### Writing Silver table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_customer")

In [0]:
%sql
select * from workspace.silver.crm_customer